# Phase 1: Data Preparation for Recommendation System

## Objective
Prepare user-item interaction data for training collaborative filtering models.

## Outputs
* `big_data.ml_features.train_interactions` - Training set (first N-2 orders per user) with:
  * **3 rating variants** to compare in model training
  * **Advanced features** (recency, co-purchase patterns)
* `big_data.ml_features.test_ground_truth` - Actual products purchased in test orders

## 🔒 Data Leakage Prevention
* Ratings calculated ONLY from train orders (not full history)
* Test data NEVER seen during feature engineering
* Temporal order strictly respected

## Strategy

### Train/Test Split (Sequential by order_number)
```
User with 10 orders:
  Train: orders 1-8 (all products)
  Test: orders 9-10 (predict these products)

User with 5 orders:
  Train: orders 1-3
  Test: orders 4-5

User with <3 orders: EXCLUDE (insufficient data)
```

### Rating Variants (3 Options to Compare)
```python
# Rating 1: Original (Frequency-based)
rating = 1.0 + (0.5 if reordered) + min(purchase_count * 0.1, 0.5)

# Rating 2: Contextual (Relationship-based)
rating_contextual = 1.0 + (recent_ratio * 0.6) + 
                    min(copurchase_count * 0.05, 0.3) +
                    (1.0 / (global_rank + 1)) * 0.1

# Rating 3: Combined (Hybrid)
rating_combined = (rating * 0.6) + (rating_contextual * 0.4)

# All ratings: Range [1.0, 2.0]
```

### Why This Approach?
* **Sequential split**: Respects temporal order (no future data leaking into training)
* **Last 2 orders as test**: Realistic scenario - predict next purchases
* **Reorder boost**: Products user repeatedly buys = strong preference signal
* **Frequency cap**: Prevents extreme ratings from dominating

---

**Expected Execution Time:** 5-10 minutes

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
import time

# Configuration
silver_schema = "big_data.silver"
ml_schema = "big_data.ml_features"

# Create ML schema if doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ml_schema}")

print("Configuration:")
print(f"  Source: {silver_schema}")
print(f"  Target: {ml_schema}")
print(f"  Ready to process!")

## Step 1: Load Source Tables

Load orders and order_products from Silver layer.

In [0]:
print("Loading Silver tables...\n")

start = time.time()

# Load orders (has order_number and user_id)
orders = spark.table(f"{silver_schema}.orders").select(
    "order_id",
    "user_id",
    "order_number",
    "order_dow",
    "order_hour_of_day"
)

# Load order_products (has product_id and reordered flag)
order_products = spark.table(f"{silver_schema}.order_products").select(
    "order_id",
    "product_id",
    "reordered"
)

print(f"Orders loaded: {orders.count():,} rows")
print(f"Order-Products loaded: {order_products.count():,} rows")
print(f"Time: {time.time() - start:.1f}s\n")

## Step 2: Build User-Item Interaction Matrix

Join orders with order_products to get user-product interactions with context.

In [0]:
print("Building user-item interaction matrix...\n")

start = time.time()

# Join to get user-product interactions with order context
interactions = order_products.join(orders, "order_id")

print(f"Total interactions: {interactions.count():,}")
print(f"Time: {time.time() - start:.1f}s\n")

# Preview
print("Sample interactions:")
interactions.select("user_id", "product_id", "order_number", "reordered").show(5)

## Step 3: Calculate User-Product Purchase Statistics

For each user-product pair, calculate:
* Total times purchased
* Times reordered
* Last order number where purchased

In [0]:
print("⚠️  NOTE: This cell will be executed AFTER train/test split to avoid data leakage.")
print("    Stats will be calculated using TRAIN data only.\n")
print("    Skipping for now...")

## Step 4: Calculate Implicit Ratings

**Rating Formula:**
```
rating = 1.0                                 # Base purchase
       + (0.5 if reordered else 0.0)        # Reorder boost
       + min(times_purchased * 0.1, 0.5)    # Frequency bonus (capped)
```

**Rationale:**
* **Base 1.0**: Every purchase is a positive signal
* **Reorder +0.5**: Repeat purchase = strong preference (not just trying it)
* **Frequency +0.1 per purchase (max 0.5)**: More purchases = stronger signal, but capped to avoid extreme values
* **Range [1.0, 2.0]**: Keeps ratings reasonable for ALS optimization

In [0]:
print("⚠️  NOTE: Ratings will be calculated AFTER train/test split.")
print("    This prevents data leakage from test set.\n")
print("    Skipping for now...")

## Step 5: Filter Users with Sufficient Data

**Minimum Requirement:** Users must have at least 3 orders.

**Why?**
* Need at least 1 order for training (order 1-3 at minimum)
* Need at least 1-2 orders for testing (order 4-5)
* Users with <3 orders: insufficient pattern data

In [0]:
print("Filtering users with sufficient order history...\n")

start = time.time()

# Find max order number per user
user_max_orders = orders.groupBy("user_id").agg(
    F.max("order_number").alias("max_order_number")
)

# Filter users with at least 3 orders
valid_users = user_max_orders.filter(F.col("max_order_number") >= 3)

total_users = user_max_orders.count()
valid_users_count = valid_users.count()
excluded = total_users - valid_users_count

print(f"Total users: {total_users:,}")
print(f"Valid users (>=3 orders): {valid_users_count:,} ({valid_users_count/total_users*100:.1f}%)")
print(f"Excluded (<3 orders): {excluded:,}")
print(f"Time: {time.time() - start:.1f}s\n")

## Step 6: Sequential Train/Test Split

**Strategy:**
* **Train**: All orders from 1 to (max_order - 2)
* **Test**: Last 2 orders (max_order - 1) and (max_order)

**Example:**
```
User with 10 orders:
  Train: orders 1, 2, 3, 4, 5, 6, 7, 8
  Test: orders 9, 10

User with 5 orders:
  Train: orders 1, 2, 3
  Test: orders 4, 5

User with 3 orders:
  Train: order 1
  Test: orders 2, 3
```

This ensures temporal order is respected (no data leakage).

In [0]:
print("Creating train/test split...\n")

start = time.time()

# Join interactions with valid users and their max order
interactions_with_max = interactions \
    .join(valid_users, "user_id") \
    .withColumn(
        "split",
        F.when(F.col("order_number") <= F.col("max_order_number") - 2, "train")
         .otherwise("test")
    )

# Separate train and test orders
train_orders = interactions_with_max.filter(F.col("split") == "train")
test_orders = interactions_with_max.filter(F.col("split") == "test")

print(f"Train interactions: {train_orders.count():,}")
print(f"Test interactions: {test_orders.count():,}")
print(f"Time: {time.time() - start:.1f}s\n")

# Show split distribution
print("Orders per split:")
interactions_with_max.groupBy("split").agg(
    F.countDistinct("user_id").alias("users"),
    F.countDistinct("order_id").alias("orders"),
    F.count("*").alias("interactions")
).show()

## Step 7: Prepare Training Interactions

For ALS training, we need:
* `user_id` (int)
* `product_id` (int)
* `rating` (double)

Join train orders with computed ratings.

In [0]:
print("Building training dataset (CLEAN - NO DATA LEAKAGE)...\n")

start = time.time()

print("Step 1: Calculate stats from TRAIN orders only...")
# ✅ CRITICAL: Calculate statistics ONLY from train_orders (not full interactions)
train_user_product_stats = train_orders.groupBy("user_id", "product_id").agg(
    F.count("*").alias("times_purchased"),
    F.sum(F.when(F.col("reordered") == True, 1).otherwise(0)).alias("times_reordered"),
    F.max("order_number").alias("last_purchased_order")
)

print(f"  Stats calculated for {train_user_product_stats.count():,} user-product pairs\n")

print("Step 2: Calculate ratings from TRAIN stats only...")
# ✅ Calculate ratings from clean train statistics
train_interactions = train_user_product_stats.withColumn(
    "rating",
    F.lit(1.0) +  # Base
    F.when(F.col("times_reordered") > 0, 0.5).otherwise(0.0) +  # Reorder boost
    F.least(F.col("times_purchased") * 0.1, F.lit(0.5))  # Frequency bonus (capped)
).withColumn(
    "_created_at", F.current_timestamp()
)

print(f"\n✅ Train interactions (CLEAN): {train_interactions.count():,}")
print(f"   Time: {time.time() - start:.1f}s\n")

# Show rating distribution
print("Rating distribution (train only):")
train_interactions.select("rating").summary("min", "25%", "50%", "75%", "max", "mean").show()

# Verify schema
print("Train interactions schema:")
train_interactions.printSchema()

# Sample
print("\nSample train data:")
train_interactions.orderBy(F.desc("rating")).show(10)

## Step 8: Calculate Advanced Contextual Features

**New Features:**
1. **Recent Purchase Ratio** - How relevant the product is in last 7 orders
2. **User Co-Purchase Count** - How often this user buys this product with others (personal basket patterns)
3. **Global Co-Purchase Rank** - How "popular" this product is in combinations (global patterns)

These features capture:
* ✅ Temporal relevance (is the user still buying it?)
* ✅ Personal basket composition (does user buy it with other products?)
* ✅ Global popularity patterns (is it commonly combined?)

In [0]:
print("Calculating Feature 1: Recent Purchase Ratio...\n")

start = time.time()

# Configuration
recent_window = 7  # Last 7 orders

# Identify recent orders (last 7 orders before test split)
user_recent_threshold = valid_users.withColumn(
    "recent_threshold",
    F.col("max_order_number") - 2 - recent_window  # -2 for test orders
)

# Filter train orders that are "recent" (within last 7 train orders)
recent_train = train_orders.join(
    user_recent_threshold.select("user_id", "recent_threshold"),
    "user_id"
).filter(
    F.col("order_number") > F.col("recent_threshold")
)

# Count purchases in recent window per user-product
recent_purchase_stats = recent_train.groupBy("user_id", "product_id").agg(
    F.count("*").alias("recent_purchases")
).withColumn(
    "recent_purchase_ratio",
    F.col("recent_purchases") / F.lit(recent_window)
)

print(f"✅ Recent purchase ratios calculated for {recent_purchase_stats.count():,} user-product pairs")
print(f"   Time: {time.time() - start:.1f}s\n")

# Sample
print("Sample recent ratios:")
recent_purchase_stats.orderBy(F.desc("recent_purchase_ratio")).show(5)

In [0]:
print("Calculating Feature 2: User-Specific Co-Purchase Count...\n")

start = time.time()

# Self-join to find products purchased together in same order
# For each product, count how many times it appeared with OTHER products
user_copurchase = train_orders.alias("a").join(
    train_orders.alias("b"),
    (F.col("a.order_id") == F.col("b.order_id")) &
    (F.col("a.user_id") == F.col("b.user_id")) &
    (F.col("a.product_id") != F.col("b.product_id")),  # Different products
    "inner"
).groupBy(
    F.col("a.user_id").alias("user_id"),
    F.col("a.product_id").alias("product_id")
).agg(
    F.count("*").alias("user_copurchase_count")
)

print(f"✅ User co-purchase counts calculated for {user_copurchase.count():,} user-product pairs")
print(f"   Time: {time.time() - start:.1f}s\n")

# Sample
print("Sample user co-purchase counts (products bought with others):")
user_copurchase.orderBy(F.desc("user_copurchase_count")).show(5)

In [0]:
print("Calculating Feature 3: Global Co-Purchase Popularity Rank...\n")

start = time.time()

# Calculate global co-purchase frequency per product
# (How often each product is bought with ANY other product across ALL users)
global_copurchase = train_orders.alias("a").join(
    train_orders.alias("b"),
    (F.col("a.order_id") == F.col("b.order_id")) &
    (F.col("a.product_id") < F.col("b.product_id")),  # Avoid duplicates
    "inner"
).groupBy(
    F.col("a.product_id").alias("product_id")
).agg(
    F.count("*").alias("global_copurchase_count")
).withColumn(
    "global_copurchase_rank",
    F.row_number().over(Window.orderBy(F.desc("global_copurchase_count")))
)

print(f"✅ Global co-purchase ranks calculated for {global_copurchase.count():,} products")
print(f"   Time: {time.time() - start:.1f}s\n")

# Sample - most "sociable" products
print("Top 10 most co-purchased products (globally):")
global_copurchase.orderBy("global_copurchase_rank").show(10)

## Step 9: Create Multiple Rating Variants

We'll create **3 different ratings** to compare in model training:

### 1. `rating` (Original - Frequency-based)
```python
rating = 1.0 + (0.5 if reordered) + min(times_purchased × 0.1, 0.5)
# Range: [1.0, 2.0]
# Captures: Purchase frequency + reorder behavior
```

### 2. `rating_contextual` (New - Relationship-based)
```python
rating_contextual = 1.0 + 
                    recent_ratio × 0.6 +
                    min(user_copurchase_count × 0.05, 0.3) +
                    (1.0 / (global_rank + 1)) × 0.1
# Range: [1.0, 2.0]
# Captures: Recency + basket patterns + global popularity
```

### 3. `rating_combined` (Hybrid)
```python
rating_combined = (rating × 0.6) + (rating_contextual × 0.4)
# Range: [1.0, 2.0]
# Captures: Best of both worlds
```

**Strategy:** Test all 3 in notebook 2 to see which performs best!

In [0]:
print("Joining all features...\n")

start = time.time()

# Join train_interactions with all new features
train_interactions_enriched = train_interactions \
    .join(recent_purchase_stats.select("user_id", "product_id", "recent_purchases", "recent_purchase_ratio"), 
          ["user_id", "product_id"], "left") \
    .join(user_copurchase.select("user_id", "product_id", "user_copurchase_count"), 
          ["user_id", "product_id"], "left") \
    .join(global_copurchase.select("product_id", "global_copurchase_count", "global_copurchase_rank"), 
          "product_id", "left")

# Fill missing values
train_interactions_enriched = train_interactions_enriched \
    .fillna(0, subset=["recent_purchases", "recent_purchase_ratio", "user_copurchase_count", "global_copurchase_count"]) \
    .fillna(99999, subset=["global_copurchase_rank"])  # Products never co-purchased = low rank

print(f"✅ All features joined: {train_interactions_enriched.count():,} rows")
print(f"   Time: {time.time() - start:.1f}s\n")

In [0]:
print("Calculating rating variants...\n")

start = time.time()

# Rating 2: Contextual (relationship-based)
train_interactions_enriched = train_interactions_enriched.withColumn(
    "rating_contextual",
    F.lit(1.0) +  # Base
    (F.col("recent_purchase_ratio") * 0.6) +  # Recency (60% weight)
    F.least(F.col("user_copurchase_count") * 0.05, F.lit(0.3)) +  # Personal basket patterns (capped at 0.3)
    (1.0 / (F.col("global_copurchase_rank") + 1)) * 0.1  # Global popularity (10% weight)
)

# Rating 3: Combined (hybrid)
train_interactions_enriched = train_interactions_enriched.withColumn(
    "rating_combined",
    (F.col("rating") * 0.6) + (F.col("rating_contextual") * 0.4)
)

print(f"✅ Rating variants calculated\n")
print(f"   Time: {time.time() - start:.1f}s\n")

# Show rating distributions
print("Rating distributions comparison:")
train_interactions_enriched.select(
    "rating", "rating_contextual", "rating_combined"
).summary("min", "25%", "50%", "75%", "max", "mean").show()

# Sample with all ratings
print("\nSample data with all ratings:")
train_interactions_enriched.orderBy(F.desc("rating_combined")).select(
    "user_id", "product_id",
    "times_purchased", "times_reordered", "recent_purchases", "user_copurchase_count",
    "rating", "rating_contextual", "rating_combined"
).show(10, truncate=False)

## Step 10: Create Test Ground Truth

For evaluation, we need to know which products each user **actually purchased** in test orders.

**Output:**
* `user_id`
* `actual_products` (array of product_ids purchased in test orders)

In [0]:
print("Creating test ground truth...\n")

start = time.time()

# Aggregate products purchased in test orders per user
test_ground_truth = test_orders.groupBy("user_id").agg(
    F.collect_set("product_id").alias("actual_products"),
    F.countDistinct("order_id").alias("test_order_count"),
    F.count("product_id").alias("total_test_products")
).withColumn(
    "_created_at", F.current_timestamp()
)

print(f"Test users: {test_ground_truth.count():,}")
print(f"Time: {time.time() - start:.1f}s\n")

# Statistics
print("Test set statistics:")
test_ground_truth.select(
    "total_test_products"
).summary("min", "25%", "50%", "75%", "max", "mean").show()

print("\nSample ground truth (first 5 users):")
test_ground_truth.orderBy("user_id").limit(5).select(
    "user_id",
    "test_order_count",
    "total_test_products",
    F.slice("actual_products", 1, 5).alias("sample_products")
).show(truncate=False)

## Step 11: Persist to Delta Tables

Save prepared datasets for model training and evaluation.

In [0]:
print("⚠️  SKIPPED: user_item_interactions table (would contain contaminated ratings)\n")
print("   We only need clean train_interactions for model training.")
print("   No need to save full history with ratings.\n")

In [0]:
print("Persisting enriched train interactions...\n")

start = time.time()

target_table = f"{ml_schema}.train_interactions"

# Save enriched version with all features and ratings
train_interactions_enriched.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

row_count = spark.table(target_table).count()

print(f"✅ Saved: {target_table} (ENRICHED)")
print(f"   Rows: {row_count:,}")
print(f"   Time: {time.time() - start:.1f}s\n")

print("Columns saved:")
for col in spark.table(target_table).columns:
    print(f"  - {col}")

In [0]:
print("Persisting test ground truth...\n")

start = time.time()

target_table = f"{ml_schema}.test_ground_truth"

test_ground_truth.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

row_count = spark.table(target_table).count()

print(f"✅ Saved: {target_table}")
print(f"   Rows: {row_count:,}")
print(f"   Time: {time.time() - start:.1f}s\n")

## Summary & Validation

Data preparation complete! Let's verify the outputs.

In [0]:
print("=" * 80)
print("✅ DATA PREPARATION COMPLETE (NO DATA LEAKAGE)")
print("=" * 80)

print("\n📊 Created Tables:\n")

tables = [
    f"{ml_schema}.train_interactions",
    f"{ml_schema}.test_ground_truth"
]

for table in tables:
    count = spark.table(table).count()
    print(f"  ✅ {table}")
    print(f"     Rows: {count:,}\n")

print("\n✨ Features Created:")
print("  ✅ rating (original) - Frequency + reordering")
print("  ✅ rating_contextual (new) - Recency + co-purchase patterns")
print("  ✅ rating_combined (hybrid) - Best of both\n")

print("📈 Advanced Features:")
print("  - recent_purchase_ratio (temporal relevance)")
print("  - user_copurchase_count (personal basket patterns)")
print("  - global_copurchase_rank (global popularity)\n")

print("\n🔒 Data Leakage Prevention: VERIFIED")
print("   - All features/ratings calculated ONLY from train orders")
print("   - Test data never seen during feature engineering")
print("   - Temporal order respected (train before test)\n")

print("=" * 80)
print("NEXT STEP: Run notebook 2_Model_Training_Recommendation")
print("  You can now test 3 different ratings to see which performs best!")
print("=" * 80)